In [5]:
import torch
import numpy as np

# 체크포인트 경로
ckpt1_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Few/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-BASELINE3/42/Embed:carte_Edge:mlp_A:gat_v1_S:42_20251204_210327.pt"
ckpt2_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Few/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-BASELINE5/42/Embed:carte_Edge:mlp_A:gat_v1_S:42_20251204_210959.pt"
def compare_tensors(t1, t2, name=""):
    """Tensor 비교 함수"""
    if t1.shape != t2.shape:
        print(f"  {name}: ❌ Different shapes: {t1.shape} vs {t2.shape}")
        return False
    
    is_equal = torch.allclose(t1, t2, rtol=1e-5, atol=1e-8)
    if not is_equal:
        max_diff = torch.abs(t1 - t2).max().item()
        mean_diff = torch.abs(t1 - t2).mean().item()
        print(f"  {name}: ❌ Values differ")
        print(f"     Shape: {t1.shape}")
        print(f"     Max difference: {max_diff:.2e}")
        print(f"     Mean difference: {mean_diff:.2e}")
        return False
    return True

def compare_dicts(d1, d2, prefix=""):
    """Dictionary 재귀 비교 함수"""
    keys1 = set(d1.keys())
    keys2 = set(d2.keys())
    
    all_same = True
    
    if keys1 != keys2:
        print(f"  {prefix}: ❌ Different keys")
        print(f"    Only in d1: {keys1 - keys2}")
        print(f"    Only in d2: {keys2 - keys1}")
        return False
    
    for key in keys1:
        val1 = d1[key]
        val2 = d2[key]
        full_key = f"{prefix}.{key}" if prefix else key
        
        if isinstance(val1, torch.Tensor):
            if not compare_tensors(val1, val2, full_key):
                all_same = False
        elif isinstance(val1, dict):
            if not compare_dicts(val1, val2, full_key):
                all_same = False
        else:
            if val1 != val2:
                print(f"  {full_key}: ❌ Values differ: {val1} vs {val2}")
                all_same = False
    
    return all_same

# 체크포인트 로드
print("Loading checkpoints...")
ckpt1 = torch.load(ckpt1_path, map_location='cpu')
ckpt2 = torch.load(ckpt2_path, map_location='cpu')

print("\n" + "="*80)
print("CHECKPOINT COMPARISON")
print("="*80)

# 1. 키 비교
keys1 = set(ckpt1.keys())
keys2 = set(ckpt2.keys())

print("\n[1] Key Comparison:")
print(f"  Checkpoint 1 keys: {len(keys1)}")
print(f"  Checkpoint 2 keys: {len(keys2)}")
print(f"  Common keys: {len(keys1 & keys2)}")
if keys1 - keys2:
    print(f"  Only in ckpt1: {keys1 - keys2}")
if keys2 - keys1:
    print(f"  Only in ckpt2: {keys2 - keys1}")

# 2. 공통 키에 대해 값 비교
print("\n[2] Value Comparison (for common keys):")
print("-"*80)

all_same = True
for key in sorted(keys1 & keys2):
    val1 = ckpt1[key]
    val2 = ckpt2[key]
    
    # 타입이 다른 경우
    if type(val1) != type(val2):
        print(f"\n{key}:")
        print(f"  ❌ Different types: {type(val1)} vs {type(val2)}")
        all_same = False
        continue
    
    # Tensor인 경우
    if isinstance(val1, torch.Tensor):
        # shape 비교
        if val1.shape != val2.shape:
            print(f"\n{key}:")
            print(f"  ❌ Different shapes: {val1.shape} vs {val2.shape}")
            all_same = False
            continue
        
        # 값 비교
        is_equal = torch.allclose(val1, val2, rtol=1e-5, atol=1e-8)
        
        if not is_equal:
            max_diff = torch.abs(val1 - val2).max().item()
            mean_diff = torch.abs(val1 - val2).mean().item()
            print(f"\n{key}:")
            print(f"  ❌ Values differ")
            print(f"     Shape: {val1.shape}")
            print(f"     Max difference: {max_diff:.2e}")
            print(f"     Mean difference: {mean_diff:.2e}")
            print(f"     Ckpt1 - mean: {val1.mean().item():.6f}, std: {val1.std().item():.6f}")
            print(f"     Ckpt2 - mean: {val2.mean().item():.6f}, std: {val2.std().item():.6f}")
            all_same = False
        else:
            print(f"{key}: ✅ Identical (shape: {val1.shape})")
    
    # Dictionary인 경우
    elif isinstance(val1, dict):
        print(f"\n{key}: (dict with {len(val1)} keys)")
        if not compare_dicts(val1, val2, key):
            all_same = False
        else:
            print(f"  ✅ All values identical")
    
    # 기타 타입
    else:
        try:
            if val1 != val2:
                print(f"\n{key}:")
                print(f"  ❌ Values differ: {val1} vs {val2}")
                all_same = False
            else:
                print(f"{key}: ✅ Identical ({type(val1).__name__})")
        except:
            print(f"\n{key}: ⚠️  Could not compare ({type(val1).__name__})")

print("\n" + "="*80)
if all_same and keys1 == keys2:
    print("✅ RESULT: Checkpoints are IDENTICAL")
else:
    print("❌ RESULT: Checkpoints are DIFFERENT")
print("="*80)

Loading checkpoints...

CHECKPOINT COMPARISON

[1] Key Comparison:
  Checkpoint 1 keys: 6
  Checkpoint 2 keys: 6
  Common keys: 6

[2] Value Comparison (for common keys):
--------------------------------------------------------------------------------

args:
  ❌ Values differ: Namespace(random_seed=42, train_epochs=1000, batch_size=32, input_dim=768, hidden_dim=192, output_dim=1, dropout_rate=0.1, source_data=['Heart_disease_statlog', 'Cardiovascular_Disease_Dataset', 'heart_target_3', 'heart_target_4'], target_data='heart', few_shot=4, num_classes=2, source_lr=0.0001, source_lr_few=1e-05, llm_model='gpt2_mean', use_gpu=True, des='BASELINE3', base_dir='test20251204', baseline=[], table_path='/storage/personal/eungyeop/dataset/table/', del_feat=[], del_exp='You did not entered the exp type', no_self_loop=False, use_target_head=True, coord_softmax_temp=0.5, coord_reg_lambda=0.2, coord_target_mode='soft', coord_tau=0.3, n_graphs=8, n_nodes=8, graph_dim=768, fgw_alpha=1, alpha=0.9, eps=0.0

In [1]:
import torch

# 파일 경로 설정
ckpt_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-BASELINE5/42/best.pt"

print(f"Loading checkpoint from: {ckpt_path} ...\n")

# CPU로 로드 (GPU 메모리 부족 방지 및 호환성 위함)
try:
    checkpoint = torch.load(ckpt_path, map_location='cpu')
    
    # 1. 최상위 키 확인 (args, epoch, model_state_dict 등이 무엇이 있는지)
    print("=== [1] Top-level Keys ===")
    print(checkpoint.keys())
    print("\n")

    # 2. 모델 파라미터(state_dict) 차원 확인
    print("=== [2] Model Parameter Dimensions ===")
    
    # 보통 'model_state_dict' 혹은 'state_dict' 키에 저장됩니다.
    # 만약 키가 없다면 checkpoint 자체가 state_dict일 수도 있습니다.
    if 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
    elif 'state_dict' in checkpoint:
        state_dict = checkpoint['state_dict']
    else:
        state_dict = checkpoint # 통째로 저장된 경우

    # 각 레이어의 이름과 쉐입 출력
    for key, tensor in state_dict.items():
        if torch.is_tensor(tensor):
            print(f"{key:<50} | Shape: {tuple(tensor.shape)}")
        else:
            print(f"{key:<50} | Value: {tensor} (Not a tensor)")

    # 3. 저장된 Argument 확인 (옵션)
    if 'args' in checkpoint:
        print("\n=== [3] Saved Arguments ===")
        print(checkpoint['args'])

except Exception as e:
    print(f"Error loading checkpoint: {e}")

Loading checkpoint from: /storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-BASELINE5/42/best.pt ...

=== [1] Top-level Keys ===
dict_keys(['model_state_dict', 'epoch', 'val_auc_mean', 'val_aucs_per_source', 'args'])


=== [2] Model Parameter Dimensions ===
basis_cls                                          | Shape: (1, 1, 768)
latent_graph.node_embeddings                       | Shape: (8, 8, 768)
latent_graph.q_proj.weight                         | Shape: (64, 768)
latent_graph.q_proj.bias                           | Shape: (64,)
latent_graph.k_proj.weight                         | Shape: (64, 768)
latent_graph.k_proj.bias                           | Shape: (64,)
gnn_experts.graph_gnns.0.linear.weight             | Shape: (192, 768)
gnn_experts.graph_gnns.0.linear.

In [7]:
import torch

ckpt1_path ="/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-GATFREEZE1/42/best.pt"
ckpt2_path = "/storage/personal/eungyeop/experiments/checkpoints/gpt2_mean/Heart_disease_statlog+Cardiovascular_Disease_Dataset+heart_target_3+heart_target_4/Pre/ngraphs-8_nnodes-8_gdim-768_nbasis-2_basis-ind_attn-gat_v1_fgw_alpha-1_vq_beta-0.3_kl_gamma-2.0_target_data-heart_description-GATFREEZE2/42/best.pt"

# 로드
ckpt1 = torch.load(ckpt1_path, map_location='cpu')
ckpt2 = torch.load(ckpt2_path, map_location='cpu')

# state_dict 추출
state1 = ckpt1.get('model_state_dict', ckpt1.get('state_dict', ckpt1))
state2 = ckpt2.get('model_state_dict', ckpt2.get('state_dict', ckpt2))

# LCG node_embeddings 비교
key = "latent_graph.node_embeddings"

if key in state1 and key in state2:
    emb1 = state1[key]
    emb2 = state2[key]
    
    print(f"Shape: {emb1.shape}")
    print(f"Are they identical? {torch.allclose(emb1, emb2)}")
    
    if not torch.allclose(emb1, emb2):
        print(f"\nMax difference: {torch.max(torch.abs(emb1 - emb2)).item():.6e}")
        print(f"Mean difference: {torch.mean(torch.abs(emb1 - emb2)).item():.6e}")
else:
    print(f"'{key}' not found!")

Shape: torch.Size([8, 8, 768])
Are they identical? False

Max difference: 1.096577e-01
Mean difference: 5.116507e-03


Loading checkpoints...

=== Searching for KMeans centroid ===
❌ KMeans centroid not found in checkpoint!

Available keys in checkpoint:
  - model_state_dict
  - epoch
  - val_auc_mean
  - val_aucs_per_source
  - args

⚠️ Note: KMeans centroid는 best.pt에 저장되지 않았을 수 있습니다.
   init_lcg 함수에서 별도로 저장해야 합니다.


{'centroids': array([[[-0.08077843,  0.17338507, -0.31364146, ...,  0.06246174,
          -0.04231519,  0.23687454],
         [-0.30334443,  0.79938704, -0.51697266, ...,  0.7972975 ,
           0.07477636,  0.31454432],
         [-0.28295764,  0.28791478, -0.4196764 , ...,  0.14958496,
           0.01928095,  0.415338  ],
         ...,
         [-0.6141687 , -0.11068259, -0.1102041 , ...,  0.04892553,
           0.2585273 ,  0.6912076 ],
         [ 0.00301297,  0.04761307, -0.10096532, ..., -0.03852982,
          -0.01825312,  0.07029235],
         [-0.0636728 ,  0.5838078 , -0.02953732, ...,  0.33784756,
          -0.00633696, -0.11570735]],
 
        [[-0.47979096,  0.14111798, -0.18113525, ...,  0.22734022,
           0.17143422,  0.55094135],
         [-0.5558056 ,  0.07417776, -0.20832668, ...,  0.16746043,
           0.2075593 ,  0.7073459 ],
         [-0.4192174 , -0.10132746, -0.11141673, ..., -0.03651989,
           0.168521  ,  0.50287545],
         ...,
         [ 0.2965921